# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, explore, and process a biomedical tabular dataset defined using the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure that `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The 'metadata' object is an mlcroissant.Metadata instance (not a dict-like object).
print(f"Loaded dataset: {dataset.metadata.name}\n\n{dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers. The Croissant metadata provides structured record sets and fields, each with unique `@id` entries.

In [ ]:
# Display record sets and their fields
record_set_ids = []
print("Record sets in this dataset and their @id:")
for rset in dataset.metadata.record_sets:
    print(f"- {rset.name} (@id: {rset.id})")
    record_set_ids.append(rset.id)
    print("  Fields:")
    for field in rset.fields:
        col_ids = ', '.join([col.id for col in getattr(field, 'columns', [])]) if hasattr(field, 'columns') and field.columns else ''
        print(f"    - {field.name} (@id: {field.id}) | Columns: {col_ids if col_ids else 'N/A'} | Type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
    print()

## 3. Data Extraction
We can extract each record set as a pandas DataFrame for analysis. Entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Collect all record sets as DataFrames
dataframes = {}

for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded record set: {recset_id} ({len(df)} records, {len(df.columns)} columns)")

# As an example, select the first record set for inspection
if record_set_ids:
    example_recset_id = record_set_ids[0]
    print(f"\nFirst record set columns: {dataframes[example_recset_id].columns.tolist()}")
    dataframes[example_recset_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's conduct some basic data processing: filtering, normalizing a numeric field, and grouping the data. All fields are referenced by their `@id`.

Below, we:
- Filter rows for a numeric field (`@id`)
- Normalize this numeric field
- Group by a categorical field (also referenced by its `@id`)

In [ ]:
# For demonstration, we'll try to find a numeric field in the first record set
recset_id = example_recset_id  # Using the first record set loaded above
df = dataframes[recset_id]

# List columns to manually select an example numeric field @id and a grouping (categorical) field @id
print("Available columns (@id) in the DataFrame:")
print(list(df.columns))

# Try to find likely numeric and group-able fields by inspecting a row
print("\nExample record:")
print(df.iloc[0])

# For demonstration, we select numeric_field_id and group_field_id below.
# Please update manually if your dataset differs.

# Guess potential numeric fields (e.g., 'http://senscience.ai/age' or similar)
# We'll use the first float/integer-like column, if available
import numpy as np
numeric_field_id = None
for col in df.columns:
    # Try converting column to numeric
    try:
        if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
            numeric_field_id = col
            break
    except Exception:
        continue

if not numeric_field_id:
    print("No numeric field detected.")
else:
    print(f"Numeric field automatically selected: {numeric_field_id}")
    threshold = df[numeric_field_id].astype(float).mean()
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical field
    # We'll pick the first column with less than 10 unique values (for demonstration)
    group_field_id = None
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        if col != numeric_field_id and n_unique > 1 and n_unique < 10:
            group_field_id = col
            break
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (using @id):")
        print(grouped.head())

## 5. Visualization

Let's visualize the distribution of the numeric field we analyzed above, and (if a grouping exists) compare group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=20, kde=True)
    plt.title(f"Distribution of Numeric Field (@id: {numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:

- Load and inspect a FAIR^2 tabular biomedical dataset with `mlcroissant`
- Reference all data elements (record sets, fields, columns) using their Croissant `@id`
- Extract, filter, normalize, group, and visualize real clinical data using pandas and seaborn/matplotlib

This approach ensures transparency and interoperability aligned with the FAIR data principles, enabling robust downstream analytics and reproducibility in medical data science workflows.